# D241 — Hive Basics: Catalogs, Tables, Views, and CTAS

This lab continues from **D240 — Introduction to Apache Hive**. It uses Hive 4.0.1 on a single-node Hadoop 3.3.6 environment with HiveServer2, a MySQL metastore, HDFS, YARN, and MapReduce.

> Run the shell cells from a notebook kernel that can access your WSL/Linux environment, or copy each `%%bash` cell's commands into a WSL terminal. The CSV files in this lesson deliberately have **no header row**, so header handling is not required.

## Learning path

1. Explore the Hive data catalog
2. Understand tables and storage
3. Create an internal (managed) table
4. Create an external table over CSV files in HDFS
5. Create a logical view
6. Compare Hive CTEs and temporary tables with temporary views
7. Try a materialized view
8. Create a table from a query with CTAS

## 0. Service spot-check

HiveServer2 should listen on port `10000`, and the metastore should listen on `9083`. The Hadoop processes should already be running.

In [ ]:
%%bash
jps
echo '--- Hive listeners ---'
ss -lnt | grep -E ':(9083|10000|10002)\b' || true
echo '--- HDFS health ---'
hdfs dfsadmin -report | grep -E 'Live datanodes|Name:'

Expected: one live DataNode plus the Hadoop and Hive Java processes. If port `10000` is absent, start the metastore first and HiveServer2 second as shown in D240.

# 1. The Hive data catalog

A **data catalog** describes data so users and query engines can discover and interpret it. Hive stores catalog metadata in the metastore database. The actual table rows normally live in HDFS.

The catalog includes:

- databases (namespaces);
- tables and whether they are managed or external;
- columns, data types, comments, and constraints;
- storage formats, SerDes, and field delimiters;
- HDFS locations;
- partitions and table properties;
- views and their defining queries.

The catalog is therefore **metadata about data**. MySQL stores this metadata; it does not store the CSV rows used in this lab.

In [ ]:
%%bash
beeline -u 'jdbc:hive2://localhost:10000/default' -n "$USER" --silent=true -e "
SHOW DATABASES;
CREATE DATABASE IF NOT EXISTS hive_basics
COMMENT 'Tables and views for the D241 Hive Basics lab';
DESCRIBE DATABASE EXTENDED hive_basics;
USE hive_basics;
SHOW TABLES;
"

`SHOW`, `DESCRIBE`, and `DESCRIBE FORMATTED` are catalog-discovery commands. `DESCRIBE FORMATTED table_name` is especially useful because it shows the table type, location, input format, SerDe, and table properties.

# 2. Tables: schema plus storage

A Hive table combines a catalog definition with files in a storage system. Defining columns does not move data by itself. The table's `LOCATION`, storage format, and row format tell Hive where the files are and how to decode them.

| Table type | Data location | Who controls data lifecycle? | Typical use |
|---|---|---|---|
| Managed/internal | Hive warehouse by default | Hive | Intermediate or Hive-owned datasets |
| External | Explicit HDFS location | Data owner outside Hive | Shared, landing, or durable source data |

`INTERNAL` is a common explanatory term. In Hive SQL, a normal `CREATE TABLE` creates a **managed table**; there is no `INTERNAL` keyword.

# 3. Internal (managed) table

We first create a small headerless CSV file locally. Each line must match the declared column order: `product_id,product_name,category,price`.

In [ ]:
%%bash
tee /tmp/d241_products.csv > /dev/null <<'EOF'
101,Laptop,Electronics,65000.00
102,Mouse,Electronics,850.00
103,Desk,Furniture,12000.00
104,Chair,Furniture,7500.00
105,Notebook,Stationery,120.00
EOF
echo 'Local headerless CSV:'
cat /tmp/d241_products.csv

Create a managed table, then use `LOAD DATA LOCAL INPATH`. `LOCAL` means the source is on the client/server's local Linux filesystem. Hive copies or moves it into the managed table's HDFS warehouse location, depending on the environment and command behavior.

In [ ]:
%%bash
beeline -u 'jdbc:hive2://localhost:10000/hive_basics' -n "$USER" --silent=true -e "
DROP TABLE IF EXISTS products_managed;
CREATE TABLE products_managed (
  product_id INT COMMENT 'Unique product identifier',
  product_name STRING,
  category STRING,
  price DECIMAL(10,2)
)
COMMENT 'Hive-owned product master'
ROW FORMAT DELIMITED
FIELDS TERMINATED BY ','
STORED AS TEXTFILE;
LOAD DATA LOCAL INPATH '/tmp/d241_products.csv'
OVERWRITE INTO TABLE products_managed;
SELECT * FROM products_managed ORDER BY product_id;
"

Inspect the catalog record. Look for `Table Type: MANAGED_TABLE`, the HDFS `Location`, the comma field delimiter, and the text input format.

In [ ]:
%%bash
beeline -u 'jdbc:hive2://localhost:10000/hive_basics' -n "$USER" --silent=true -e "
DESCRIBE products_managed;
DESCRIBE FORMATTED products_managed;
"

You can also locate the managed files from HDFS. The database directory usually ends in `.db`.

In [ ]:
%%bash
hdfs dfs -find /user/hive/warehouse -name 'products_managed' -print
hdfs dfs -ls /user/hive/warehouse/hive_basics.db/products_managed 2>/dev/null || true

# 4. External table over HDFS CSV files

For an external table, prepare the data independently, upload it to a durable HDFS directory, and then register that location in the Hive catalog. The CSV below is headerless and has the columns `order_id,customer_name,city,amount,order_date`.

In [ ]:
%%bash
tee /tmp/d241_orders.csv > /dev/null <<'EOF'
1001,Asha,Bengaluru,1250.00,2026-08-15
1002,Ravi,Chennai,850.00,2026-08-15
1003,Meera,Bengaluru,2100.00,2026-08-16
1004,Arun,Hyderabad,1750.00,2026-08-16
1005,Diya,Chennai,950.00,2026-08-17
1006,Kiran,Bengaluru,3200.00,2026-08-17
EOF
echo 'Local headerless CSV:'
cat /tmp/d241_orders.csv

The cleanup below is intentionally limited to this lab's exact HDFS directory. It makes the upload repeatable and prevents duplicate files.

In [ ]:
%%bash
hdfs dfs -rm -r -f /user/hive/external/d241_orders
hdfs dfs -mkdir -p /user/hive/external/d241_orders
hdfs dfs -put /tmp/d241_orders.csv /user/hive/external/d241_orders/
hdfs dfs -ls /user/hive/external/d241_orders
hdfs dfs -cat /user/hive/external/d241_orders/d241_orders.csv

Register the directory as an external table. Because the file has no header, every line is a data row and no header-related table property is needed.

In [ ]:
%%bash
beeline -u 'jdbc:hive2://localhost:10000/hive_basics' -n "$USER" --silent=true -e "
DROP TABLE IF EXISTS orders_external;
CREATE EXTERNAL TABLE orders_external (
  order_id INT,
  customer_name STRING,
  city STRING,
  amount DECIMAL(10,2),
  order_date DATE
)
ROW FORMAT DELIMITED
FIELDS TERMINATED BY ','
STORED AS TEXTFILE
LOCATION '/user/hive/external/d241_orders';
SELECT * FROM orders_external ORDER BY order_id;
"

In [ ]:
%%bash
beeline -u 'jdbc:hive2://localhost:10000/hive_basics' -n "$USER" --silent=true -e "
DESCRIBE FORMATTED orders_external;
SELECT city, COUNT(*) AS order_count, SUM(amount) AS total_amount
FROM orders_external
GROUP BY city
ORDER BY city;
"

Look for `Table Type: EXTERNAL_TABLE` and the explicit HDFS location. Expected city totals are Bengaluru 6550, Chennai 1800, and Hyderabad 1750.

### Prove the external-table lifecycle

Dropping an external table removes its catalog definition but should leave its data files in HDFS. We will drop and recreate only the metadata.

In [ ]:
%%bash
beeline -u 'jdbc:hive2://localhost:10000/hive_basics' -n "$USER" --silent=true -e "
DROP TABLE orders_external;
SHOW TABLES LIKE 'orders_external';
"
echo 'The catalog entry is gone, but the external file remains:'
hdfs dfs -ls /user/hive/external/d241_orders

Recreate the catalog entry over the same unchanged data.

In [ ]:
%%bash
beeline -u 'jdbc:hive2://localhost:10000/hive_basics' -n "$USER" --silent=true -e "
CREATE EXTERNAL TABLE orders_external (
  order_id INT, customer_name STRING, city STRING,
  amount DECIMAL(10,2), order_date DATE
)
ROW FORMAT DELIMITED FIELDS TERMINATED BY ','
STORED AS TEXTFILE
LOCATION '/user/hive/external/d241_orders';
SELECT COUNT(*) AS restored_row_count FROM orders_external;
"

# 5. Views

A normal Hive **view** stores a query definition in the catalog. It does not store a separate copy of the result. Each query against the view reads its underlying tables. Views can simplify repeated SQL and expose only selected columns or rows.

In [ ]:
%%bash
beeline -u 'jdbc:hive2://localhost:10000/hive_basics' -n "$USER" --silent=true -e "
DROP VIEW IF EXISTS bengaluru_orders_v;
CREATE VIEW bengaluru_orders_v AS
SELECT order_id, customer_name, amount, order_date
FROM orders_external
WHERE city = 'Bengaluru';
SHOW VIEWS;
DESCRIBE FORMATTED bengaluru_orders_v;
SELECT * FROM bengaluru_orders_v ORDER BY order_id;
"

The view always reflects current rows from `orders_external`. If the base table is dropped or changed incompatibly, the view can become invalid.

## 5A. Does Hive have temporary views?

Apache Hive does **not** provide Spark SQL's session-scoped `CREATE TEMP VIEW` feature. In Hive, choose one of these alternatives:

- **CTE (`WITH ...`)** for a named result used only within one statement;
- **temporary table** for data reused during one Hive session;
- **regular view** for a saved query definition shared through the metastore.

A Hive temporary table exists only in the current session, is not visible to other sessions, is automatically removed when the session ends, and can shadow a permanent table with the same name. Hive stores temporary-table data in a session-specific scratch location rather than managing it as a normal shared catalog object.

### Query-scoped alternative: CTE

The name `large_orders` exists only while this single statement runs. No catalog object or stored result is created.

In [ ]:
%%bash
beeline -u 'jdbc:hive2://localhost:10000/hive_basics' -n "$USER" --silent=true -e "
WITH large_orders AS (
  SELECT order_id, customer_name, city, amount
  FROM orders_external
  WHERE amount >= 1500
)
SELECT * FROM large_orders ORDER BY order_id;
"

### Session-scoped alternative: temporary table

The temporary table and its query must run through the **same Beeline connection**. Because the `-e` invocation below is one session, `recent_orders_temp` exists for the `SELECT` and disappears when Beeline exits.

In [ ]:
%%bash
beeline -u 'jdbc:hive2://localhost:10000/hive_basics' -n "$USER" --silent=true -e "
CREATE TEMPORARY TABLE recent_orders_temp
STORED AS ORC
AS
SELECT order_id, customer_name, city, amount, order_date
FROM orders_external
WHERE order_date >= '2026-08-16';
SELECT * FROM recent_orders_temp ORDER BY order_id;
"
echo 'Beeline exited: its session-scoped temporary table has now been removed.'

Do not expect `recent_orders_temp` to exist in the next notebook cell: each `beeline -e` command opens a new session. To experiment interactively, connect once with Beeline, create the temporary table, run several queries, and then use `!quit`.

# 6. Materialized views

A **materialized view** stores the query result physically. Reading a precomputed summary can be faster than recomputing it, but the stored result must be maintained or rebuilt when base data changes. Hive may also use eligible materialized views for automatic query rewriting.

> Materialized-view behavior depends on Hive version and configuration. Hive 4 supports the syntax below, but a local installation may restrict creation, rewriting, or rebuild operations. Treat this section as an optional capability check—not a prerequisite for the rest of the lab.

In [ ]:
%%bash
beeline -u 'jdbc:hive2://localhost:10000/hive_basics' -n "$USER" --silent=true -e "
DROP MATERIALIZED VIEW IF EXISTS city_sales_mv;
CREATE MATERIALIZED VIEW city_sales_mv
DISABLE REWRITE
AS
SELECT city, COUNT(*) AS order_count, SUM(amount) AS total_amount
FROM orders_external
GROUP BY city;
SELECT * FROM city_sales_mv ORDER BY city;
"

`DISABLE REWRITE` keeps this first example focused on stored results rather than automatic optimizer substitution. After base data changes, rebuild with:

```sql
ALTER MATERIALIZED VIEW city_sales_mv REBUILD;
```

If creation fails because the local setup does not support the required materialized-view features, record the error, skip this optional object, and continue to CTAS.

# 7. CREATE TABLE AS SELECT (CTAS)

CTAS creates a new table from a query and writes the result into it. Unlike a logical view, the result is physically stored. Unlike a materialized view, a CTAS table is simply a snapshot and has no automatic relationship to its source afterward.

This example creates a managed ORC summary table. ORC is a columnar format designed for analytics and is generally preferable to CSV for curated Hive tables.

In [ ]:
%%bash
beeline -u 'jdbc:hive2://localhost:10000/hive_basics' -n "$USER" --silent=true -e "
DROP TABLE IF EXISTS city_sales_ctas;
CREATE TABLE city_sales_ctas
STORED AS ORC
AS
SELECT city, COUNT(*) AS order_count, SUM(amount) AS total_amount
FROM orders_external
GROUP BY city;
SELECT * FROM city_sales_ctas ORDER BY city;
DESCRIBE FORMATTED city_sales_ctas;
"

Look for `MANAGED_TABLE`, an ORC input format, and a warehouse location. If new rows later appear in `orders_external`, they do not automatically appear in `city_sales_ctas`; rerun the CTAS workflow to create a new snapshot.

## Catalog inventory

Use catalog commands to review everything created in this lesson.

In [ ]:
%%bash
beeline -u 'jdbc:hive2://localhost:10000/hive_basics' -n "$USER" --silent=true -e "
SHOW TABLES;
SHOW VIEWS;
SHOW MATERIALIZED VIEWS;
"

If `SHOW MATERIALIZED VIEWS` is unsupported in your build, use `SHOW TABLES` and `DESCRIBE FORMATTED city_sales_mv` after successful creation.

# 8. Key comparisons

| Object | Stores definition? | Stores separate result data? | Updates automatically when queried? |
|---|---:|---:|---:|
| External table | Yes | Uses existing external files | Reads current files |
| Managed table | Yes | Yes, Hive-owned | Reads current table files |
| View | Yes | No | Re-runs its query |
| CTE | Only inside one statement | No | Re-evaluated within its statement |
| Temporary table | Only in one session | Yes, in session scratch storage | Ends with the session |
| Materialized view | Yes | Yes | Requires maintenance/rebuild |
| CTAS table | Yes | Yes | No; it is a snapshot |

The central lesson is that the **catalog definition** and the **data files** are separate concerns. Table type determines who owns the files; views and CTAS determine whether a query is saved as logic or materialized as data.

## Optional cleanup

Keep the objects if later lessons will reuse them. To remove only the D241 lab objects, run the SQL below. The external HDFS data is intentionally retained unless you explicitly remove its exact lab directory.

```sql
USE hive_basics;
DROP MATERIALIZED VIEW IF EXISTS city_sales_mv;
DROP VIEW IF EXISTS bengaluru_orders_v;
DROP TABLE IF EXISTS city_sales_ctas;
DROP TABLE IF EXISTS products_managed;
DROP TABLE IF EXISTS orders_external;
```

Dropping `orders_external` does not remove `/user/hive/external/d241_orders`. Delete that HDFS directory only when you no longer need the source data.